# Notebook 5 – Évaluation et Benchmark

**PFE 2025/2026** – Segmentation d'images en présence de bruit  

---

Ce notebook présente :
1. Les métriques d'évaluation (IoU, Dice, Précision, Rappel, F1)
2. Le benchmark de toutes les pipelines
3. Analyse comparative des résultats
4. Conclusions et recommandations

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from pathlib import Path
from evaluation import compute_iou, compute_dice, compute_precision_recall_f1, evaluate_mask
from evaluation import benchmark_noise_levels
from segmentation.classical import segment_classical
from denoising import denoise
from utils.visualization import plot_all_metrics, save_figure

print('Modules chargés ✓')

## 5.1 Métriques d'évaluation – Définitions

In [ ]:
# Illustration des métriques sur un exemple simple
size = 100
gt   = np.zeros((size, size), dtype=np.uint8)
pred = np.zeros((size, size), dtype=np.uint8)

cv2.circle(gt,   (50, 50), 35, 255, -1)  # Vérité terrain
cv2.circle(pred, (55, 50), 35, 255, -1)  # Prédiction (légèrement décalée)

m = evaluate_mask(pred, gt)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))

axes[0].imshow(gt,   cmap='gray'); axes[0].set_title('Vérité terrain (GT)'); axes[0].axis('off')
axes[1].imshow(pred, cmap='gray'); axes[1].set_title('Prédiction');          axes[1].axis('off')

# Overlay
overlay = np.zeros((size, size, 3), dtype=np.uint8)
overlay[np.logical_and(gt>127, pred>127)] = [0, 255, 0]    # TP vert
overlay[np.logical_and(gt>127, pred<=127)] = [255, 0, 0]   # FN rouge
overlay[np.logical_and(gt<=127, pred>127)] = [0, 0, 255]   # FP bleu
axes[2].imshow(overlay); axes[2].set_title('TP=vert FN=rouge FP=bleu'); axes[2].axis('off')

# Métriques
axes[3].axis('off')
text = '\n'.join([f'{k.upper():12s}: {v:.4f}' for k, v in m.items()])
axes[3].text(0.1, 0.5, text, transform=axes[3].transAxes,
             fontfamily='monospace', fontsize=11, va='center')

plt.suptitle('Illustration des métriques de segmentation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/metriques_illustration.png', dpi=150, bbox_inches='tight')
plt.show()
print(m)

## 5.2 Benchmark complet

In [ ]:
# Générer données synthétiques de test
def make_pairs(n=10, size=256):
    rng = np.random.default_rng(0)
    images, masks = [], []
    for _ in range(n):
        img  = np.full((size, size, 3), 40, dtype=np.uint8)
        mask = np.zeros((size, size), dtype=np.uint8)
        cx   = rng.integers(size//4, 3*size//4)
        cy   = rng.integers(size//4, 3*size//4)
        r    = rng.integers(size//6, size//3)
        cv2.circle(img,  (cx, cy), r, (180, 180, 180), -1)
        cv2.circle(mask, (cx, cy), r, 255,             -1)
        images.append(img)
        masks.append(mask)
    return images, masks

# Utiliser les vraies données si disponibles
from utils import load_dataset
images, masks_gt, names = load_dataset('../data/images', '../data/masks')
if not images:
    print('Données synthétiques utilisées (aucune image dans data/)')
    images, masks_gt = make_pairs(n=10)
else:
    print(f'{len(images)} images chargées')

In [ ]:
# Définir les pipelines à comparer
pipelines = {
    'Canny brut':        {'segmentor': lambda img: segment_classical(img, 'canny'),
                          'denoising_method': None},
    'Canny + Médian':    {'segmentor': lambda img: segment_classical(img, 'canny'),
                          'denoising_method': 'median'},
    'Canny + Bilatéral': {'segmentor': lambda img: segment_classical(img, 'canny'),
                          'denoising_method': 'bilateral'},
    'Otsu brut':         {'segmentor': lambda img: segment_classical(img, 'otsu'),
                          'denoising_method': None},
    'Otsu + Médian':     {'segmentor': lambda img: segment_classical(img, 'otsu'),
                          'denoising_method': 'median'},
    'Otsu + NLM':        {'segmentor': lambda img: segment_classical(img, 'otsu'),
                          'denoising_method': 'nlm'},
}

# Benchmark Gaussien
print('Benchmark bruit Gaussien...')
df_gauss = benchmark_noise_levels(
    images, masks_gt, pipelines,
    noise_type='gaussian',
    noise_levels=[0, 10, 25, 50, 75]
)

# Benchmark Sel-et-Poivre
print('Benchmark bruit Sel-et-Poivre...')
df_sp = benchmark_noise_levels(
    images, masks_gt, pipelines,
    noise_type='salt_and_pepper',
    noise_levels=[0, 0.02, 0.05, 0.10, 0.20]
)

print('Benchmark terminé ✓')

In [ ]:
# Tableau récapitulatif
summary_g = df_gauss.groupby(['pipeline', 'noise_level'])[['iou', 'dice', 'f1']].mean().round(3)
print('=== Benchmark Gaussien ===')
print(summary_g.to_string())

In [ ]:
# Figures IoU vs bruit Gaussien
from utils.visualization import plot_metrics_vs_noise

for metric in ['iou', 'dice', 'f1']:
    fig = plot_metrics_vs_noise(df_gauss, metric=metric, noise_type='gaussian', figsize=(9, 5))
    plt.savefig(f'../results/figures/{metric}_vs_gaussien.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Figure 4-panneaux complète
from utils.visualization import plot_all_metrics

fig = plot_all_metrics(df_gauss, noise_type='gaussian')
plt.savefig('../results/figures/benchmark_gaussien_complet.png', dpi=150, bbox_inches='tight')
plt.show()

fig = plot_all_metrics(df_sp, noise_type='salt_and_pepper')
plt.savefig('../results/figures/benchmark_sp_complet.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.3 Heatmap comparative (meilleure pipeline par bruit)

In [ ]:
pivot = df_gauss.groupby(['pipeline', 'noise_level'])['iou'].mean().unstack()

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_xlabel('Niveau de bruit σ')
ax.set_ylabel('Pipeline')
ax.set_title('IoU moyen par pipeline et niveau de bruit Gaussien')
plt.tight_layout()
plt.savefig('../results/figures/heatmap_iou_gaussien.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sauvegarder les résultats
df_gauss.to_csv('../results/metrics/benchmark_gaussian.csv', index=False)
df_sp.to_csv('../results/metrics/benchmark_sp.csv', index=False)
print('Résultats sauvegardés dans results/metrics/')

print('\n=== CONCLUSION ===')
best_g = df_gauss.groupby('pipeline')['iou'].mean().idxmax()
best_s = df_sp.groupby('pipeline')['iou'].mean().idxmax()
print(f'Meilleure pipeline (Gaussien) : {best_g}')
print(f'Meilleure pipeline (S&P)      : {best_s}')